# Belt Conveyor: geared drive, sensor vs computer vision

Saskatchewan Polytechnic PROJ 613, Group 9 (Ganesh Mishra, Lakshya Sharma, Sagar Patel). **Week 4.**

| | Sensor branch | Computer-vision branch |
|---|---|---|
| Data | CORA v2.1 T1 temperature | CORA v2.1 infrared camera |
| Classes | healthy vs gear-worn | healthy vs gear-worn |
| Test units | 16 runs, each speed held out in turn | the same 16 runs |
| Script | `Models/belt_conveyor_cora_baseline.py` | same script, infrared branch (OpenCV) |

CORA is one controlled motor/gearbox **laboratory rig**, used as a component analogue for a **geared** conveyor drive reducer. It says nothing about belt cracks, idlers or gearless drives. Both modalities come from the **same rig and the same runs**, but the two models are trained and scored separately; there is no fusion. The model classifies present gear condition, not a repair date.

In [1]:
DATA = [
    ("Sensor: CORA T1 samples (cache)", "Processed_Data/cora_gear_wear/t1_bounded_samples.npz", "*", 1),
    ("CV: CORA infrared stacks (16 MAT)", "Raw_Data/cora_gear_wear", "*.mat", 16),
]

import contextlib
import io
import json
import os
import platform
import runpy
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "Models").is_dir() and (p / "Preprocessing").is_dir())
RAW, PROCESSED, RESULTS = ROOT / "Raw_Data", ROOT / "Processed_Data", ROOT / "Results"


def package(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"


try:
    commit = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    commit = "unknown"
record = {"git commit": commit, "python": platform.python_version()}
record.update({n: package(n) for n in ("numpy", "scipy", "pandas", "scikit-learn", "opencv-python", "matplotlib")})
display(pd.Series(record, name="environment").to_frame())

rows = []
for label, path, pattern, expected in DATA:
    p = ROOT / path
    files = [f for f in p.rglob(pattern) if f.is_file()] if p.is_dir() else ([p] if p.is_file() else [])
    rows.append({"data": label, "location": path, "files": f"{len(files)}/{expected}",
                 "MB": round(sum(f.stat().st_size for f in files) / 1e6, 1), "local": len(files) >= expected})
availability = pd.DataFrame(rows).set_index("data")
display(availability)
LOCAL = availability["local"].to_dict()

if all(LOCAL.values()):
    for name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
        os.environ[name] = "http://127.0.0.1:9"  # offline guard: any network call now fails loudly
    os.environ["NO_PROXY"] = ""
    print("All data is local: running fully offline.")
else:
    print("Some data is missing: those steps are skipped. Download routes are in Documentation/capstone_details.md.")


def run_script(relative_path, *args, tail=3):
    """Run a project script exactly as `python <script>` would and show its last output lines."""
    path, buffer, saved = ROOT / relative_path, io.StringIO(), sys.argv
    sys.argv = [path.name, *args]
    try:
        with contextlib.redirect_stdout(buffer):
            runpy.run_path(str(path), run_name="__main__")
    finally:
        sys.argv = saved
    text = buffer.getvalue().replace(str(ROOT) + os.sep, "").replace(str(ROOT), ".")
    print(f"[{relative_path}]")
    for line in [l for l in text.splitlines() if l.strip()][-tail:]:
        print("  " + line)


def load(name):
    path = RESULTS / name
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None


def metrics(block):
    keys = ("accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1")
    return {k: round(block[k], 3) for k in keys if isinstance(block.get(k), (int, float))}


def check(before, after, paths, name):
    def pick(doc, path):
        for key in path:
            doc = doc[key]
        return metrics(doc)
    same = before is not None and all(pick(before, p) == pick(after, p) for p in paths)
    print(f"{name}: {'matches' if same else 'DIFFERS from'} the committed result")

,environment
git commit,d12c74f
python,3.11.9
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2


,location,files,MB,local
data,,,,
Sensor: CORA T1 samples (cache),Processed_Data/cora_gear_wear/t1_bounded_sampl...,1/1,1.7,True
CV: CORA infrared stacks (16 MAT),Raw_Data/cora_gear_wear,16/16,390.5,True


All data is local: running fully offline.


## Run the experiment

Target: healthy (`H`) vs gear-worn (`W25`, `W50`, `W75`), 16 stationary runs at four speeds (F5, F15, F50, F60). Each speed is held out in turn and each run gets one prediction. One script trains both branches on the same folds.

In [2]:
before = load("belt_conveyor_cora.json")
if all(LOCAL.values()):
    run_script("Models/belt_conveyor_cora_baseline.py")
belt = load("belt_conveyor_cora.json")
check(before, belt, [("sensor", "model"), ("infrared", "model"), ("sensor", "most_frequent_control")], "Belt Conveyor result")

[Models/belt_conveyor_cora_baseline.py]
  Extracted run 15/16: W75_F50_S
  Extracted run 16/16: W75_F60_S
  Saved Results\belt_conveyor_cora.json, Results\belt_conveyor_sensor_visual_comparison.md, and Figures\belt_conveyor_cora_confusions.png
Belt Conveyor result: matches the committed result


## 1. Sensor branch: T1 temperature

Eight bounded windows of each run's T1 temperature CSV (about 4.3% of its bytes, read with checksummed byte ranges) are summarised into one feature vector per run.

In [3]:
display(pd.DataFrame({
    "T1 temperature model (16 runs)": metrics(belt["sensor"]["model"]),
    "Most-frequent class (control)": metrics(belt["sensor"]["most_frequent_control"]),
}).T)

,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
T1 temperature model (16 runs),0.625,0.583,0.567,0.583,0.564
Most-frequent class (control),0.750,0.500,0.375,0.500,0.429


## 2. Computer-vision branch: infrared with OpenCV

All 450 infrared frames of each run are processed with OpenCV into one feature vector per run.

In [4]:
display(pd.DataFrame({
    "Infrared / OpenCV model (16 runs)": metrics(belt["infrared"]["model"]),
    "Most-frequent class (control)": metrics(belt["infrared"]["most_frequent_control"]),
}).T)

,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
Infrared / OpenCV model (16 runs),0.50,0.417,0.433,0.417,0.418
Most-frequent class (control),0.75,0.500,0.375,0.500,0.429


## 3. Sensor vs computer vision

In [5]:
comparison = pd.DataFrame([
    {"branch": "Sensor: T1 temperature", **metrics(belt["sensor"]["model"])},
    {"branch": "CV: infrared", **metrics(belt["infrared"]["model"])},
    {"branch": "Most-frequent class (control)", **metrics(belt["sensor"]["most_frequent_control"])},
]).set_index("branch")[["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1"]]
display(comparison)

,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
branch,,,,,
Sensor: T1 temperature,0.625,0.583,0.567,0.583,0.564
CV: infrared,0.500,0.417,0.433,0.417,0.418
Most-frequent class (control),0.750,0.500,0.375,0.500,0.429


![Belt Conveyor sensor vs CV confusion matrices](../Figures/belt_conveyor_cora_confusions.png)

**Interpretation.** Both learnt models score **below the 0.750 majority-class accuracy** (sensor 0.625, infrared 0.500), and infrared catches only one of four healthy runs. With 16 runs from one rig and one run per wear state and speed, this is a completed, leakage-aware comparison with **weak generalization**, not an effective fault detector. Full write-up: [`Results/belt_conveyor_sensor_visual_comparison.md`](../Results/belt_conveyor_sensor_visual_comparison.md).

## Conclusion

The Belt Conveyor comparison is honest but weak: on this small same-rig dataset, neither temperature nor infrared beats always guessing "worn". More runs per condition, or the full sensor signals instead of bounded samples, would be needed before drawing a modality conclusion.

**Data citation:** CORA v2.1, DOI 10.34810/DATA2500, CC BY 4.0.